# Examples — an end-to-end tour

This notebook samples synthetic marketing worlds, inspects everything inside them, plots several time series, and **validates** that the reported decomposition is exact. It only needs `prior_generator`, so you can [download it](index.ipynb) and run it yourself.

> Every cell below is executed when the docs are built — the outputs and figures you see are real.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')          # hide harmless tqdm/ipywidgets notices

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

import prior_generator as pg
from prior_generator import viz          # the plotting submodule
print('prior-generator', pg.__version__)

## 1 · Sample a world

We draw from the `confounded_spend` scenario — latent demand drives both spend and the baseline, the classic MMM confounding. Sampling is deterministic in `(cfg, seed)`.

In [ ]:
sc = pg.SCENARIOS[1]                       # confounded_spend
scm = pg.sample_scm(sc.prior(T=104, seed=0), seed=0,
                    connect_all=sc.connect_all,
                    name=sc.name, purpose=sc.purpose)

print(f'world {scm.name!r}: K={scm.K} channels, M={scm.M} controls, '
      f'J={scm.J} demand, T={scm.T} weeks')

## 2 · Access everything inside the `SCM`

`.data` holds the 13 named series, `.g` the DAG blocks, `.params` the drawn coefficients and mechanisms.

In [ ]:
print('data series:')
for name, arr in scm.data.items():
    print(f'  {name:<28} {arr.shape}')

print('\nDAG blocks (arrow counts):')
for name, block in scm.g.items():
    print(f'  {name:<7} {int(np.asarray(block).sum())}')

In [ ]:
# Per-channel mechanisms and the direct-response coefficients
from prior_generator.worlds import channel_role, mechanism_label
for k in range(scm.K):
    print(f'C{k+1}: role={channel_role(scm.g, k):<7} '
          f'mechanism={mechanism_label(scm.params, k):<20} '
          f'beta={scm.params["beta"][k]:+.2f}')

## 3 · Read the world's story

`describe_scm` renders the DAG with coefficients, node connectivity, mechanisms, the decomposition-identity check, and signal metrics as plain text.

In [ ]:
print(pg.describe_scm(scm))

## 4 · The causal graph

The package renders four figures. They write to a path, so we render to a temp file and display it inline.

In [ ]:
import tempfile, os
def show_fig(plot_fn, world, title=None):
    path = os.path.join(tempfile.mkdtemp(), 'fig.png')
    plot_fn(world, path, title or world.name)
    display(Image(filename=path))

show_fig(viz.plot_dag, scm)

## 5 · The observable time series

What a modeller actually sees: per-channel spend, the observed controls and (latent) demand, and sales.

In [ ]:
show_fig(viz.plot_timeseries, scm)

### Plot the series yourself

`.data` is plain numpy, so you can plot any slice directly. Here is each channel's spend on its own axes.

In [ ]:
weeks = np.arange(scm.T)
fig, ax = plt.subplots(figsize=(10, 4))
for k in range(scm.K):
    ax.plot(weeks, scm.data['channels'][:, k], lw=1.4, label=f'C{k+1}')
ax.set(title='Media spend by channel', xlabel='week', ylabel='spend')
ax.legend(ncol=scm.K, frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.show()

## 6 · Per-channel spend vs its true contribution

Because we simulated the world, we know each channel's *true* contribution to sales — not an estimate. Indexed to mean 1, you can see how spend sweeps its (adstocked, saturated) response.

In [ ]:
show_fig(viz.plot_channels, scm)

## 7 · The exact decomposition

Sales equals the sum of its true components to float precision. Let's prove it, then visualise every piece.

In [ ]:
d = scm.data
lhs = d['sales']
rhs = d['baseline'] + d['contributions'].sum(1) + d['indirect_effects']
print('sales = baseline + Σ direct + indirect')
print('  max |error| =', f'{np.abs(lhs - rhs).max():.2e}')

# the fully-unrolled identity == SCM.reconstruction()
print('reconstruction() == sales, max err =', f'{scm.identity_error():.2e}')

In [ ]:
show_fig(viz.plot_decomposition, scm)

### A stacked view of where sales come from

The average sales dollar, split into baseline, direct media, and the three indirect sources — a compact custom time-series view built from `.data`.

In [ ]:
src = d['indirect_effects_by_source']       # (T, 3): cc, zc, dc
layers = {
    'baseline': d['baseline'],
    'direct media': d['contributions'].sum(1),
    'indirect · cc': src[:, 0],
    'indirect · zc': src[:, 1],
    'indirect · dc': src[:, 2],
}
fig, ax = plt.subplots(figsize=(10, 4))
ax.stackplot(weeks, *layers.values(), labels=list(layers), alpha=0.9)
ax.plot(weeks, d['sales'], color='k', lw=1.2, label='sales Y')
ax.set(title='Sales, decomposed', xlabel='week', ylabel='sales')
ax.legend(loc='upper left', ncol=3, frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
plt.show()

## 8 · Different worlds, different time series

Each scenario isolates a different causal pathway. Overlaying their sales series (indexed to mean 1) shows how varied the generated dynamics are.

In [ ]:
fig, axes = plt.subplots(len(pg.SCENARIOS), 1, figsize=(10, 9), sharex=True)
for idx, ax in enumerate(axes):
    sc = pg.SCENARIOS[idx]
    w = pg.sample_scm(sc.prior(T=104, seed=0), seed=0,
                      connect_all=sc.connect_all, name=sc.name)
    y = w.data['sales']
    ax.plot(np.arange(w.T), y / y.mean(), lw=1.3)
    ax.set_ylabel(f'{idx}: {sc.name}', fontsize=8, rotation=0, ha='right', va='center')
    ax.spines[['top', 'right']].set_visible(False)
axes[-1].set_xlabel('week')
fig.suptitle('Sales across the five scenarios (indexed to mean 1)')
fig.tight_layout()
plt.show()

## 9 · Generate and validate a corpus

Scale up to a corpus — a dict of stacked arrays over many worlds. `DataGenerator.validate_corpus` checks the schema; the additive identity holds across every task.

In [ ]:
cfg = pg.make_scm_prior(n_treatments=5, n_covariates=3, n_latent=2,
                        edge_budget={'cy': (4, 4), 'dc': (2, 2), 'zc': (1, 2)},
                        n_cells=2, draws_per_cell=2, seed=42)
corpus = pg.sample_prior_predictive(cfg)
print('N tasks:', corpus['spend_raw'].shape[0])

from prior_generator.data_generator import DataGenerator
print('validation errors:', DataGenerator.validate_corpus(corpus) or 'none — OK')

In [ ]:
lhs = corpus['sales_raw'].astype(np.float64)
rhs = (corpus['baseline_raw'] + corpus['contributions_raw'].sum(-1)
       + corpus['indirect_effects']).astype(np.float64)
print('additive identity across the whole corpus:')
print('  max abs error =', f'{np.abs(lhs - rhs).max():.2e}')

### Signal diagnostics

Every corpus embeds a signal summary so weak-signal priors are caught at generation time. (This demo corpus is tiny, so a gate row may read FAIL purely from small-sample noise.)

In [ ]:
from prior_generator.signal_diagnostics import check_signal_gate
ok, lines = check_signal_gate(corpus['diagnostics']['signal'])
for line in lines:
    print(line)

## 10 · Persist a corpus and a bundle

`save_corpus` / `load_corpus` round-trip a compressed `.npz`; `write_scm_bundle` writes a human-auditable folder.

In [ ]:
import tempfile, os
root = tempfile.mkdtemp()

npz = os.path.join(root, 'corpus.npz')
pg.save_corpus(corpus, npz)
loaded = pg.load_corpus(npz)
print('corpus round-trips exactly:',
      bool(np.array_equal(corpus['spend_raw'], loaded['spend_raw'])))

bundle = pg.write_scm_bundle(scm, os.path.join(root, 'world'))
print('bundle files:', sorted(os.listdir(bundle)))

## Recap

From one seed you drew a world, reached every observable and every truth series inside it, plotted the graph and several time series, **proved** the decomposition is exact, then scaled up to a validated corpus and persisted both a corpus and an auditable bundle.

Next: the [API reference](reference/index.md) documents every public symbol, generated from the source.